# Phishing URL Detection Using Machine Learning
**Mini Project | Lab Execution + Crop Notebook Style**

This notebook follows the simple, sequential execution style used in the ML lab practicals and the section-by-section structure of the Smart Crop Yield Prediction notebook.

**Goal:** classify a URL as phishing or legitimate using four classification models.

> Label assumption used in this notebook: `0 = Phishing`, `1 = Legitimate`. Confirm this mapping from your dataset/source before presenting results.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings("ignore")

## 2. Load the Dataset

In [ ]:
# If using Google Colab, upload your CSV when asked.
from google.colab import files
uploaded = files.upload()

In [ ]:
# Change the filename below if your uploaded CSV has a different name.
df = pd.read_csv("PhiUSIIL_10000_rows_clean.csv")
df.head()

## 3. Basic Dataset Inspection

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.describe()

## 4. Check Missing Values and Duplicates

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum())

print("\nNumber of duplicate rows:", df.duplicated().sum())

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)

## 5. Check the Target Column

In [ ]:
print("Target column values:")
print(df["label"].value_counts(dropna=False))

print("\nTarget percentages:")
print((df["label"].value_counts(normalize=True, dropna=False) * 100).round(2))

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="label")
plt.title("Distribution of Phishing and Legitimate URLs")
plt.xlabel("Label (check mapping: 0 = phishing, 1 = legitimate)")
plt.ylabel("Count")
plt.show()

## 6. Simple EDA and Correlation Analysis

In [ ]:
numeric_df = df.select_dtypes(include=np.number)

print("Numeric columns:")
print(numeric_df.columns.tolist())

numeric_df.hist(figsize=(14, 10), bins=20)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), cmap="coolwarm", annot=False)
plt.title("Correlation Heatmap")
plt.show()

## 7. Data Cleaning and Feature Selection

In [ ]:
# Remove rows where the target label is missing
df = df.dropna(subset=["label"]).copy()

# Keep only the two expected classes
df = df[df["label"].isin([0, 1])].copy()

# Remove URL/text identifier columns if they exist.
# This notebook uses numeric URL features for the ML models.
columns_to_drop = ["label", "URL", "Domain"]
columns_to_drop = [col for col in columns_to_drop if col in df.columns]

X = df.drop(columns=columns_to_drop)
y = df["label"].astype(int)

# Keep numeric features, as used in the lab's basic sklearn workflow
X = X.select_dtypes(include=np.number)

print("Feature data shape:", X.shape)
print("Target data shape:", y.shape)
print("Features used:")
print(X.columns.tolist())

In [ ]:
print("Missing values in features before imputation:")
print(X.isnull().sum().sum())

print("\nMissing values in target:")
print(y.isnull().sum())

## 8. Split the Dataset (80:20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

## 9. Handle Missing Values and Scale Features

In [ ]:
# Fit the imputer only on training data to avoid data leakage
imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Fit the scaler only on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Preprocessing completed.")
print("Training data shape:", X_train_scaled.shape)
print("Testing data shape:", X_test_scaled.shape)

## 10. Model 1 — Logistic Regression

In [ ]:
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train_scaled, y_train)

logistic_pred = logistic_model.predict(X_test_scaled)

print("Logistic Regression Accuracy:", accuracy_score(y_test, logistic_pred))
print(classification_report(y_test, logistic_pred, zero_division=0))

## 11. Model 2 — Support Vector Machine (SVM)

In [ ]:
svm_model = SVC(kernel="rbf")
svm_model.fit(X_train_scaled, y_train)

svm_pred = svm_model.predict(X_test_scaled)

print("SVM Accuracy:", accuracy_score(y_test, svm_pred))
print(classification_report(y_test, svm_pred, zero_division=0))

## 12. Model 3 — Decision Tree

In [ ]:
decision_tree_model = DecisionTreeClassifier(random_state=42, max_depth=10)
decision_tree_model.fit(X_train_scaled, y_train)

decision_tree_pred = decision_tree_model.predict(X_test_scaled)

print("Decision Tree Accuracy:", accuracy_score(y_test, decision_tree_pred))
print(classification_report(y_test, decision_tree_pred, zero_division=0))

## 13. Model 4 — Random Forest

In [ ]:
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
random_forest_model.fit(X_train_scaled, y_train)

random_forest_pred = random_forest_model.predict(X_test_scaled)

print("Random Forest Accuracy:", accuracy_score(y_test, random_forest_pred))
print(classification_report(y_test, random_forest_pred, zero_division=0))

## 14. Compare All Four Models

In [ ]:
models = {
    "Logistic Regression": logistic_pred,
    "SVM": svm_pred,
    "Decision Tree": decision_tree_pred,
    "Random Forest": random_forest_pred
}

results = []

for model_name, predictions in models.items():
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision (Phishing)": precision_score(y_test, predictions, pos_label=0, zero_division=0),
        "Recall (Phishing)": recall_score(y_test, predictions, pos_label=0, zero_division=0),
        "F1-score (Phishing)": f1_score(y_test, predictions, pos_label=0, zero_division=0)
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1-score (Phishing)", ascending=False)

print("Model Performance Comparison:")
display(results_df)

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x="Model", y="Accuracy")
plt.title("Accuracy Comparison of Four Models")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x="Model", y="F1-score (Phishing)")
plt.title("Phishing F1-score Comparison")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.show()

## 15. Confusion Matrix for the Best Model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_predictions = models[best_model_name]

print("Best model based on phishing F1-score:", best_model_name)
print("\nClassification Report:")
print(classification_report(
    y_test, best_predictions,
    labels=[0, 1],
    target_names=["Phishing (0)", "Legitimate (1)"],
    zero_division=0
))

cm = confusion_matrix(y_test, best_predictions, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Phishing (0)", "Legitimate (1)"]
)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - " + best_model_name)
plt.show()

## 16. Actual vs Predicted Values

In [ ]:
comparison = pd.DataFrame({
    "Actual Label": y_test.values,
    "Predicted Label": best_predictions
})
print(comparison.head(15))

## 17. Conclusion

In [ ]:
print("Model comparison completed.")
print("Best model based on phishing F1-score:", best_model_name)
print("Review the comparison table and confusion matrix before writing your final conclusion.")

## Important Notes
- Run all cells from top to bottom.
- If the CSV filename differs, update the filename in Section 2.
- This notebook expects a numeric `label` column containing `0` and `1`, and numeric engineered URL features.
- Verify the label mapping using your dataset documentation. The metrics marked “Phishing” treat label `0` as phishing.
- This notebook follows the lab/crop notebook's simple sequential execution style. The Streamlit/Flask UI and literature review/report are separate mini-project deliverables.